# **DATA PREPROCESSING**

---

## **1. Import libraries**

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.visualization import plot_distribution_with_zoom
from src.preprocessing import detect_outliers_iqr_summary

## **2. Load Raw Data**

Trong giai đoạn đầu của quá trình tiền xử lý, nhóm tiến hành tải tập dữ liệu thô (raw data) đã được thu thập từ E-commerce. Dữ liệu này được lưu trữ dưới định dạng tệp CSV nằm trong cấu trúc thư mục dự án (`data/raw`).

In [2]:
tiki_path = "../data/raw/products.csv"
df_tiki = pd.read_csv(tiki_path)

ebay_path = "../data/raw/ebay_products.csv"
df_ebay = pd.read_csv(ebay_path)

print(f"Tiki dataset shape: {df_tiki.shape}")
print(f"Ebay dataset shape: {df_ebay.shape}")

Tiki dataset shape: (10000, 11)
Ebay dataset shape: (15744, 31)


Dữ liệu được đọc từ tệp `products.csv` và `ebay_products.csv` và được khởi tạo dưới dạng cấu trúc bảng lưu vào biến DataFrame `df_tiki` và `df_ebay`.

## **3. Data Cleaning & Consistency Check**

Quá trình thu thập dữ liệu bằng Web Crawling hoặc gọi API thường không tránh khỏi việc thu thập phải các bản ghi trùng lặp hoặc bị lỗi cấu trúc trang dẫn đến các dòng trống. Do đó, bước làm sạch và kiểm tra tính nhất quán là cực kỳ quan trọng.

### **3.1. Handling Duplicates & Empty Rows**

Để đánh giá tính toàn vẹn dữ liệu, nhóm tiến hành kiểm tra các bản ghi trùng lặp trên toàn bộ tập dữ liệu bằng hàm `duplicated()` trong Pandas. 
* **Duplicate Rows:** Kiểm tra mọi hàng có nội dung hoàn toàn giống nhau trên tất cả các thuộc tính.
* **Empty Rows:** Phát hiện các dòng bị rỗng hoàn toàn (tất cả các cột đều là giá trị NaN).

In [3]:
# Check for duplicate rows in Tiki
duplicate_mask_tiki = df_tiki.duplicated(keep=False)
num_duplicate_rows_tiki = df_tiki.duplicated().sum()
print(f"Are there any duplicate rows in Tiki?: {duplicate_mask_tiki.any()}")
print(f"Total number of duplicate rows in Tiki: {num_duplicate_rows_tiki}")

# Check for duplicate rows in Ebay
duplicate_mask_ebay = df_ebay.duplicated(keep=False)
num_duplicate_rows_ebay = df_ebay.duplicated().sum()
print(f"Are there any duplicate rows in Ebay?: {duplicate_mask_ebay.any()}")
print(f"Total number of duplicate rows in Ebay: {num_duplicate_rows_ebay}")

Are there any duplicate rows in Tiki?: False
Total number of duplicate rows in Tiki: 0
Are there any duplicate rows in Ebay?: False
Total number of duplicate rows in Ebay: 0


In [4]:
# 2. Check for completely empty rows
num_empty_rows_tiki = df_tiki.isna().all(axis=1).sum()
print(f"Total number of completely empty rows in Tiki: {num_empty_rows_tiki}")

num_empty_rows_ebay = df_ebay.isna().all(axis=1).sum()
print(f"Total number of completely empty rows in Ebay: {num_empty_rows_ebay}")

df_tiki.dropna(how='all', inplace=True)
df_ebay.dropna(how='all', inplace=True)

Total number of completely empty rows in Tiki: 0
Total number of completely empty rows in Ebay: 0


Bộ dữ liệu không chứa dòng trùng lặp hay dòng rỗng hoàn toàn, cho thấy quá trình thu thập dữ liệu (Crawl/API) đã đảm bảo tốt tính toàn vẹn và duy nhất của các bản ghi. Dữ liệu hiện đã sẵn sàng để tiến hành kiểm tra chuyên sâu hơn (xử lý Missing Values và chuẩn hóa Data Types).

### **3.2. Data Type Conversion**

Ép kiểu dữ liệu cho chuẩn xác (ví dụ: chuyển cột ngày tháng sang `datetime`, cột giá tiền từ `string` sang `float`, ID sang `category`).

Quá trình ép kiểu dữ liệu đảm bảo các biến số học không bị nhầm lẫn thành chuỗi và các cột ngày tháng (`datetime`) được đưa về đúng định dạng để phục vụ cho việc tính toán chuỗi thời gian. Các trường định danh (ID) cũng được chuyển sang dạng chuỗi (String) để tránh việc thuật toán vô tình thực hiện các phép toán thống kê trên chúng.

In [5]:
# Tiki Data Type Conversion 
# Convert IDs to string to prevent unintended mathematical operations
df_tiki['id'] = df_tiki['id'].astype(str)

# Ensure financial metrics are numeric (coerce errors to NaN)
if 'price' in df_tiki.columns:
    df_tiki['price'] = pd.to_numeric(df_tiki['price'], errors='coerce')
if 'original_price' in df_tiki.columns:
    df_tiki['original_price'] = pd.to_numeric(df_tiki['original_price'], errors='coerce')

# Convert categorical variables to string
df_tiki['brand'] = df_tiki['brand'].astype(str)
df_tiki['category'] = df_tiki['category'].astype(str)


# Ebay Data Type Conversion
df_ebay['item_id'] = df_ebay['item_id'].astype(str)

if 'price' in df_ebay.columns:
    df_ebay['price'] = pd.to_numeric(df_ebay['price'], errors='coerce')
if 'shipping_cost' in df_ebay.columns:
    df_ebay['shipping_cost'] = pd.to_numeric(df_ebay['shipping_cost'], errors='coerce')

# Convert date strings to robust datetime objects (UTC)
if 'item_creation_date' in df_ebay.columns:
    df_ebay['item_creation_date'] = pd.to_datetime(df_ebay['item_creation_date'], errors='coerce')
if 'item_end_date' in df_ebay.columns:
    df_ebay['item_end_date'] = pd.to_datetime(df_ebay['item_end_date'], errors='coerce')

In [6]:
# Output validation
print("\tTiki Data Types")
print(df_tiki[['id', 'price', 'original_price', 'rating_average', 'brand']].dtypes)

	Tiki Data Types
id                 object
price               int64
original_price      int64
rating_average    float64
brand              object
dtype: object


In [7]:
# Output validation
print("\tEbay Data Types")
print(df_ebay[['item_id', 'price', 'shipping_cost', 'item_creation_date']].dtypes)

	Ebay Data Types
item_id                            object
price                             float64
shipping_cost                     float64
item_creation_date    datetime64[ns, UTC]
dtype: object


Quá trình ép kiểu đã diễn ra thành công. Các cột ID hiện đã được bảo vệ dưới dạng chuỗi (object/string) để tránh các phép tính sai lệch. Các cột chi phí (`price`, `shipping_cost`) đã ở định dạng số (int64/float64) sẵn sàng cho các phép toán thống kê. Đặc biệt, các cột thời gian của eBay đã được chuyển sang chuẩn `datetime64 (UTC)`, tạo tiền đề vững chắc để nhóm phân tích yếu tố thời gian và vòng đời sản phẩm.

### **3.3. Structural Errors & Consistency**

Dữ liệu thu thập từ các sàn thương mại điện tử thường xuyên gặp vấn đề về tính nhất quán của văn bản (văn bản chứa khoảng trắng thừa, sai lệch chữ hoa/chữ thường, hoặc các giá trị rỗng được điền bằng các chuỗi giả mạo như 'nan', 'Root', 'OEM'). Bước này sử dụng các hàm xử lý chuỗi vector hóa của Pandas để làm sạch, đồng nhất cấu trúc dữ liệu, phục vụ tốt cho việc gom nhóm và hiển thị trên Interactive Dashboard sau này.

In [8]:
# Tiki Standardization
# Use vectorized string operations (.str) for better performance over apply(lambda)
df_tiki['brand'] = df_tiki['brand'].str.strip()
df_tiki['name'] = df_tiki['name'].str.strip()
df_tiki['category'] = df_tiki['category'].str.strip().str.title()

# Standardize pseudo-null strings into a consistent 'Unknown' category
pseudo_nulls = ['Nan', 'None', 'Unknown', 'Root', 'Oem']
# Convert brand to Title Case first to ensure accurate matching when replacing
df_tiki['brand'] = df_tiki['brand'].str.title().replace(pseudo_nulls, 'Unknown')
df_tiki['category'] = df_tiki['category'].replace(pseudo_nulls, 'Unknown')

# Ebay Standardization
# Ebay text cleansing
df_ebay['title'] = df_ebay['title'].str.strip()

if 'category_path' in df_ebay.columns:
    df_ebay['category_path'] = df_ebay['category_path'].str.strip().str.title()

if 'seller_username' in df_ebay.columns:
    # Convert seller usernames to lowercase for exact matching
    df_ebay['seller_username'] = df_ebay['seller_username'].str.strip().str.lower()
    df_ebay['seller_username'] = df_ebay['seller_username'].fillna('unknown')

In [9]:
# Output validation: Display the cleaned subsets
print("\tTiki Cleaned Sample")
display(df_tiki[['id', 'name', 'brand', 'category']].head(2))

	Tiki Cleaned Sample


,id,name,brand,category
0,277305929,Đầm suông nữ vải Linen rút eo cổ V tay ngắn da218,Arctic Hunter,Thời Trang Nữ
1,275767409,BỘ 12 CỜ LÊ VÒNG MIỆNG 6-24MM/6-32MM INGCO - H...,Ingco,Unknown


In [10]:
# Output validation: Display the cleaned subsets
print("\tEbay Cleaned Sample")
display(df_ebay[['item_id', 'title', 'seller_username', 'category_path']].head(2))

	Ebay Cleaned Sample


,item_id,title,seller_username,category_path
0,v1|276709583737|0,"Acer Swift Go 14"" Laptop Intel Core Ultra 7 15...",acer,Pc Laptops & Netbooks
1,v1|256696935114|0,"HP ProBook Touchscreen Laptop 11.6"" Core i3 8G...",discountcomputerdepot,Pc Laptops & Netbooks


Kết quả in ra cho thấy dữ liệu văn bản đã được chuẩn hóa đồng nhất. Các khoảng trắng thừa đã bị loại bỏ, định dạng chữ (viết hoa/viết thường) được đồng bộ. Đặc biệt, các giá trị rác gây nhiễu phân loại như "Root" trong cột `category` của Tiki đã được thuật toán nhận diện và quy chuẩn về "Unknown". Điều này đảm bảo khi đưa dữ liệu lên các bộ lọc (filters) của công cụ trực quan hóa, các danh mục sẽ hiển thị gọn gàng và không bị phân mảnh.

## **4. Handling Missing Values**

### **4.1. Missing Data Identification**

Tính toán tỷ lệ missing values cho từng cột (dùng bảng hoặc biểu đồ missing matrix).

# ---- Missing Data Identification ----


In [ ]:
print("Missing percentages in Tiki (%)")
print((df_tiki.isnull().sum() / len(df_tiki) * 100).sort_values(ascending=False))
print("\nMissing percentages in Ebay (%)")
print((df_ebay.isnull().sum() / len(df_ebay) * 100).sort_values(ascending=False))

### **4.2. Imputation Strategy & Execution**

* `Mean` / `Median` / `k-NN`: Phù hợp cho các biến định lượng nhỏ hoặc có phân phối chuẩn.
* `MICE (Multiple Imputation by Chained Equations)`: Là thuật toán nội suy ưu việt nhất cho **biến định lượng**.
* **Categorical Data:** Mặc dù MICE rất mạnh mẽ, nhưng việc ép MICE chạy trên các biến danh xưng đòi hỏi phải mã hóa thành số, vô tình tạo ra các mối quan hệ thứ tự giả tạo và làm sai lệch ý nghĩa xã hội học của dữ liệu. Do đó, **Mode Imputation** (Điền giá trị xuất hiện nhiều nhất) là chiến lược an toàn và chuẩn mực nhất cho nhóm biến phân loại, giúp bảo toàn nguyên vẹn tính danh xưng mà không làm nhiễu không gian đặc trưng.

Các chiến lược áp dụng:
- Drop các cột thiếu thông tin trên 90% (ví dụ như `item_end_date` bị khuyết 99.8%).
- `Mode imputation` để điền cho dữ liệu Categorical của Ebay.
- Cột giá trị đếm bằng số (`Median` / `0`).

In [ ]:
from src.preprocessing import impute_categorical_mode

# ---- Imputation Tiki ----
# Fill categorical columns with 'Unknown'
df_tiki['brand'].fillna("Unknown", inplace=True)
df_tiki['category'].fillna("Unknown", inplace=True)
# Num features median / zero fill for counts
for col in ['discount_rate', 'rating_average', 'review_count', 'quantity_sold']:
    if col in df_tiki.columns:
        df_tiki[col].fillna(0, inplace=True)

# ---- Imputation Ebay ----
# Drop > 90% missing
cols_to_drop = ['subtitle', 'item_end_date'] # item_end_date missed 99.8%
df_ebay.drop(columns=[c for c in cols_to_drop if c in df_ebay.columns], inplace=True)

# Mode Imputation cho các biến phân loại của Ebay
categorical_ebay_cols = ['shipping_currency', 'item_location_postal', 'item_location_country', 'condition_id']
df_ebay = impute_categorical_mode(df_ebay, categorical_ebay_cols)

# Fill sellers and categories missing names
df_ebay['seller_username'].fillna("Unknown", inplace=True)
df_ebay['category_path'].fillna('Unknown', inplace=True)
if 'condition' in df_ebay.columns:
    df_ebay['condition'].fillna('Unknown', inplace=True)

# Fill missing numericals with zero or median
if 'shipping_cost' in df_ebay.columns:
    df_ebay['shipping_cost'].fillna(0, inplace=True)
if 'seller_feedback_score' in df_ebay.columns:
    df_ebay['seller_feedback_score'].fillna(df_ebay['seller_feedback_score'].median(), inplace=True)
if 'seller_feedback_percent' in df_ebay.columns:
    df_ebay['seller_feedback_percent'].fillna(df_ebay['seller_feedback_percent'].median(), inplace=True)

## **5. Outlier Detection & Treatment**

### **5.1. Visualizing Distributions**

Để phát hiện ngoại lai (outliers) và hiểu rõ hình dáng phân phối của các biến định lượng, nhóm tiến hành trực quan hóa bằng cách kết hợp hai loại biểu đồ: *Boxplot* và *Histogram*.

Do đặc thù của dữ liệu thương mại điện tử thường có độ lệch phải rất mạnh (bởi một số ít sản phẩm có giá trị hoặc lượt bán cực kỳ cao), việc trực quan hóa toàn bộ dữ liệu sẽ khiến biểu đồ bị bóp nghẹt. Để khắc phục, nhóm áp dụng kỹ thuật "Zoom-in", tạm thời lọc bỏ 5% dữ liệu cực đoan nhất ở đuôi phải (`quantile_threshold=0.95`) khi vẽ đồ thị. Cách tiếp cận này giúp giữ lại cấu trúc phân phối cốt lõi của đại đa số sản phẩm mà vẫn đảm bảo tính thẩm mỹ và dễ đọc.

In [ ]:
numeric_features_tiki = ['price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold']
numeric_features_ebay = ['price', 'shipping_cost', 'seller_feedback_score', 'seller_feedback_percent']

print("----- TIKI DISTRIBUTIONS -----")
plot_distribution_with_zoom(
    df=df_tiki,
    columns=[c for c in numeric_features_tiki if c in df_tiki.columns],
    quantile_threshold=0.95
)

print("----- EBAY DISTRIBUTIONS -----")
plot_distribution_with_zoom(
    df=df_ebay,
    columns=[c for c in numeric_features_ebay if c in df_ebay.columns],
    quantile_threshold=0.95
)

### **5.2. Statistical Detection**

Để xác định chính xác ranh giới của các điểm ngoại lai bằng con số cụ thể, nhóm áp dụng phương pháp thống kê *IQR (Interquartile Range)* thay vì Z-score. Lý do là vì Z-score rất nhạy cảm với các giá trị cực đoan và chỉ thực sự chính xác khi dữ liệu có phân phối chuẩn. Trong khi đó, dữ liệu E-commerce của chúng ta chủ yếu là phân phối lệch phải, do đó IQR – phương pháp dựa trên các tứ phân vị (Q1, Q3) và trung vị – sẽ mang lại độ chuẩn xác và tính bền vững cao hơn.

Công thức xác định ranh giới:
* **IQR** = $Q3 - Q1$
* **Lower Bound** = $Q1 - 1.5 \times IQR$
* **Upper Bound** = $Q3 + 1.5 \times IQR$

Những điểm dữ liệu nằm ngoài khoảng $[Lower\_Bound, Upper\_Bound]$ sẽ được thống kê là ngoại lai (Outliers).

In [ ]:
outlier_summary_tiki = detect_outliers_iqr_summary(
    df=df_tiki,
    columns=[c for c in numeric_features_tiki if c in df_tiki.columns]
)
print("OUTLIER SUMMARY (IQR METHOD) - TIKI:")
display(outlier_summary_tiki)

outlier_summary_ebay = detect_outliers_iqr_summary(
    df=df_ebay,
    columns=[c for c in numeric_features_ebay if c in df_ebay.columns]
)
print("\nOUTLIER SUMMARY (IQR METHOD) - EBAY:")
display(outlier_summary_ebay)

Dựa vào bảng thống kê IQR, bộ dữ liệu mang đặc trưng lệch phải điển hình của ngành thương mại điện tử. 

* **Nhóm biến giá (price, original_price):** Tỷ lệ ngoại lai chiếm khoảng 12%. Việc ranh giới dưới mang giá trị âm là hệ quả toán học do dữ liệu bị lệch phải nặng (giá tiền không thể âm). Ranh giới trên (~829,000 - 967,000 VND) cho thấy các sản phẩm phân khúc cao cấp đang bị thuật toán gắn mác là ngoại lai.
* **Nhóm biến bán hàng & tương tác (quantity_sold, review_count):** Có tỷ lệ ngoại lai cao nhất (~16.6%). Ranh giới trên rất thấp (lượt bán > 229 hoặc số đánh giá > 25 đã bị coi là ngoại lai). Thực chất, đây chính là các sản phẩm "Best-seller" hoặc sản phẩm xu hướng – nhóm dữ liệu có giá trị kinh doanh cao nhất.
* **Nhóm biến tỷ lệ & thang đo (discount_rate, rating_average):** Điểm đánh giá giới hạn từ 0-5 nên hoàn toàn an toàn (0% ngoại lai). Tỷ lệ giảm giá có 2.8% ngoại lai (lớn hơn 50%), đây có thể là các sản phẩm Flash Sale xả kho hoặc lỗi thiết lập giá từ nhà bán hàng.

### **5.3. Outlier Handling**

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

# ---- Xử lý ngoại lai cho Tiki (Sử dụng Isolation Forest trên giá trị price & original_price) ----
features_tiki = ['price', 'original_price', 'quantity_sold']
features_tiki = [f for f in features_tiki if f in df_tiki.columns]
df_tiki_clean = df_tiki.dropna(subset=features_tiki).copy()

if features_tiki:
    iso_tiki = IsolationForest(contamination=0.05, random_state=42)
    tiki_outlier_preds = iso_tiki.fit_predict(df_tiki_clean[features_tiki])
    # Chỉ giữ lại inliers (pred == 1)
    df_tiki_clean = df_tiki_clean[tiki_outlier_preds == 1]
    
# ---- Xử lý ngoại lai cho Ebay (IQR Capping cho price và Isolation Forest) ----
features_ebay = ['price', 'seller_feedback_score']
features_ebay = [f for f in features_ebay if f in df_ebay.columns]
df_ebay_clean = df_ebay.dropna(subset=features_ebay).copy()

if features_ebay:
    iso_ebay = IsolationForest(contamination=0.05, random_state=42)
    ebay_outlier_preds = iso_ebay.fit_predict(df_ebay_clean[features_ebay])
    df_ebay_clean = df_ebay_clean[ebay_outlier_preds == 1]

print(f"Số lượng bản ghi Tiki (ban đầu): {df_tiki.shape[0]} => (Sau iForest): {df_tiki_clean.shape[0]}")
print(f"Số lượng bản ghi Ebay (ban đầu): {df_ebay.shape[0]} => (Sau iForest): {df_ebay_clean.shape[0]}")

## **6. Feature Engineering (Tạo biến mới)**

Để đáp ứng chính xác các mục tiêu phân tích, nhóm bắt buộc phải dùng Pandas để sinh ra các biến mới mang ý nghĩa kinh doanh. Bằng việc tính toán những đặc trưng nâng cao trực tiếp từ các cột thô ban đầu, dữ liệu đã sẵn sàng để vẽ biểu đồ và phân tích chuyên sâu bên trong Dashboard.

### **6.1. Cho Tiki (Thành viên 1 & 4):**
- **Discount_Segment**: Chia `discount_rate` thành các nhóm Categorical (ví dụ: "0%", "< 20%", "20-50%", "> 50%") để dễ dàng vẽ Boxplot hoặc Bar Chart so sánh lượt bán.
- **Is_Best_Seller**: Biến nhị phân (1/0) dựa trên `quantity_sold` (ví dụ: > 100 là 1) để phân lớp sản phẩm phổ biến.

### **6.2. Cho eBay (Thành viên 2 & 3):**
- **Total_Cost_VND**: Chuyển đổi `price_usd` + `shipping_cost_usd` sang tiền Việt (nhân tỷ giá 25,000) để đồng bộ thang đo với Tiki khu phân tích Dashboard.
- **Listing_Duration_Days**: Lấy `item_end_date` trừ đi `item_creation_date` để tính vòng đời bài đăng. Do cột này bị mất 99% dữ liệu nên không thể khai thác chuyên sâu nhưng vẫn tiến hành tạo biến đối với những bản ghi có sẵn.
- **Trust_Level**: Nhóm `seller_feedback_percent` thành "High Trust" (>98%) và "Normal/Low Trust" (<98%) để phân tích mức độ uy tín ảnh hưởng đến chênh lệch giá.

In [ ]:
from src.preprocessing import engineer_tiki_features, engineer_ebay_features

print("Tạo biến mới cho dữ liệu Tiki...")
df_tiki_clean = engineer_tiki_features(df_tiki_clean)
display(df_tiki_clean[['price', 'discount_rate', 'quantity_sold', 'Discount_Segment', 'Is_Best_Seller']].head())

print("Tạo biến mới cho dữ liệu Ebay...")
df_ebay_clean = engineer_ebay_features(df_ebay_clean)
display(df_ebay_clean[['price', 'shipping_cost', 'seller_feedback_percent', 'Total_Cost_VND', 'Trust_Level']].head())

## **7. Data Structuring (Normalization & Splitting into 5 Tables)**

Bước này sẽ dùng Pandas để chuẩn hóa và chia 2 file ban đầu thành 5 DataFrames đại diện cho 5 bảng có quan hệ với nhau (cho dashboard sau này).
1. `Dim_Product`: Thông tin sản phẩm chung (id, tên, thương hiệu, danh mục, nền tảng)
2. `Fact_Listings_Tiki`: Biến động giá, lượt bán, đánh giá, mức giảm giá trên Tiki (Bao gồm Data Engineering Features)
3. `Fact_Listings_Ebay`: Biến động giá, phí ship, tình trạng sản phẩm trên Ebay (Kèm theo Data Engineering Features)
4. `Dim_Seller`: Bảng thông tin người bán trên Ebay (Bao gồm Trust_Level)
5. `Dim_Category`: Danh mục sản phẩm (hoặc có thể dùng làm lookup category chung)

In [ ]:
# --- Table 1: Dim_Product ---
# Tiki attributes
dim_prod_tiki = df_tiki_clean[['id', 'name', 'brand', 'category']].copy()
dim_prod_tiki.rename(columns={'id': 'product_id', 'name': 'product_name'}, inplace=True)
dim_prod_tiki['platform'] = 'Tiki'

# Ebay attributes
dim_prod_ebay = df_ebay_clean[['item_id', 'title', 'category_path']].copy()
dim_prod_ebay.rename(columns={'item_id': 'product_id', 'title': 'product_name', 'category_path': 'category'}, inplace=True)
dim_prod_ebay['brand'] = 'Unknown'
dim_prod_ebay['platform'] = 'Ebay'

# Ngộp lại hai mảng thành Dim_Product chung và xóa trùng
dim_product = pd.concat([dim_prod_tiki, dim_prod_ebay], ignore_index=True)
dim_product = dim_product.drop_duplicates(subset=['product_id', 'platform'])
print(f"Dim_Product shape: {dim_product.shape}")

# --- Table 2: Dim_Seller ---
if 'seller_username' in df_ebay_clean.columns:
    seller_cols = ['seller_username', 'seller_feedback_score', 'seller_feedback_percent', 'Trust_Level']
    dim_seller = df_ebay_clean[[c for c in seller_cols if c in df_ebay_clean.columns]].copy()
    dim_seller = dim_seller.drop_duplicates(subset=['seller_username'])
else:
    dim_seller = pd.DataFrame()
print(f"Dim_Seller shape: {dim_seller.shape}")

# --- Table 3: Fact_Listings_Tiki ---
# Snapshot values representing price points, reviews and Features of Tiki products
tiki_fact_cols = ['id', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'Discount_Segment', 'Is_Best_Seller']
fact_tiki = df_tiki_clean[[c for c in tiki_fact_cols if c in df_tiki_clean.columns]].copy()
fact_tiki.rename(columns={'id': 'product_id'}, inplace=True)
print(f"Fact_Listings_Tiki shape: {fact_tiki.shape}")

# --- Table 4: Fact_Listings_Ebay ---
ebay_fact_cols = ['item_id', 'seller_username', 'price', 'shipping_cost', 'Total_Cost_VND', 'condition', 'item_creation_date', 'Listing_Duration_Days']
fact_ebay = df_ebay_clean[[c for c in ebay_fact_cols if c in df_ebay_clean.columns]].copy()
fact_ebay.rename(columns={'item_id': 'product_id'}, inplace=True)
print(f"Fact_Listings_Ebay shape: {fact_ebay.shape}")

# --- Table 5: Dim_Category ---
dim_category = dim_product[['category']].copy().drop_duplicates().reset_index(drop=True)
dim_category['category_id'] = ['CAT_' + str(i+1).zfill(4) for i in range(len(dim_category))]
print(f"Dim_Category shape: {dim_category.shape}")

## **8. Export Processed Data**

Lưu các bảng đã được làm sạch và chuẩn hóa ra file CSV vào thư mục `data/processed/`. Các file này sẽ được sử dụng để xây dựng Dashboard trên Streamlit ở các bước sau.

In [ ]:
import os

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

# Lấy lại category_id cho dim_product trước khi lưu
dim_product = pd.merge(dim_product, dim_category, on='category', how='left')
dim_product.drop(columns=['category'], inplace=True)

# Export to CSV
dim_product.to_csv(os.path.join(processed_dir, "dim_product.csv"), index=False)
dim_seller.to_csv(os.path.join(processed_dir, "dim_seller.csv"), index=False)
fact_tiki.to_csv(os.path.join(processed_dir, "fact_tiki_listings.csv"), index=False)
fact_ebay.to_csv(os.path.join(processed_dir, "fact_ebay_listings.csv"), index=False)
dim_category.to_csv(os.path.join(processed_dir, "dim_category.csv"), index=False)

print("Đã lưu 5 file `.csv` thành công vào thư mục:", processed_dir)
print("Các file gồm:", os.listdir(processed_dir))